In [3]:
import open3d as o3d
import numpy as np

# Đường dẫn file
ply_path = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Test/ply/0266.ply"
pcd = o3d.io.read_point_cloud(ply_path)

In [4]:
bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound=(-0.6, -0.3, -1.3),
                                            max_bound=(0.5, 0.4, -1.0))
pcd_crop = pcd.crop(bbox)

In [ ]:

# === 2. Convert coordinate system (z<0 → z>0) ===
R_convert = np.array([[1, 0, 0],
                      [0, -1, 0],
                      [0, 0, -1]])
pcd_crop.rotate(R_convert, center=(0, 0, 0))

# === 3. Ground truth pose từ CSV ===
gt_pos = np.array([-0.122, 0.062, 0.983])   # x, y, z
gt_dir = np.array([0.429, -0.025, -0.903]) # normal vector
gt_dir /= np.linalg.norm(gt_dir)  # Chuẩn hóa

# === 4. Tạo arrow hiển thị hướng ===
arrow = o3d.geometry.TriangleMesh.create_arrow(
    cone_radius=0.01,
    cone_height=0.03,
    cylinder_radius=0.005,
    cylinder_height=0.05
)
arrow.paint_uniform_color([1, 0, 0])  # màu đỏ

# Căn chỉnh mũi tên với vector gt_dir
z_axis = np.array([0, 0, 1])
v = np.cross(z_axis, gt_dir)
c = np.dot(z_axis, gt_dir)
if np.linalg.norm(v) < 1e-6:  # trùng hướng
    R = np.eye(3)
else:
    vx = np.array([[0, -v[2], v[1]],
                   [v[2], 0, -v[0]],
                   [-v[1], v[0], 0]])
    R = np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v)**2))

arrow.rotate(R, center=(0, 0, 0))
arrow.translate(gt_pos)

# === 5. Visualize ===
o3d.visualization.draw_geometries([pcd_crop, arrow])


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


: 